7.1 RAG(검색 증강 생성)의 성능을 좌우하는 ㄴ요소
- 하나의 질의로부터 적절한 문서를 찾아내는 검색기의 성능
- 검색된 결과를 바탕으로 답변을 생성하는 답변 단계에서 사용하는 거대 언어 모델의 성능

7.2 리랭킹(ReRanking)
- 초기에 검색된 결과를 더 정밀하게 평가하고, 이를 기반으로 결과의 순위를 재조정
- 검색어의 관련성을 평가하여 관련성 점수(relevance score)계산 후 재정렬
   .relevance score : 0~1사이 값, 1에 가까울수록 관련성이 높음
7.2.1. LLM 기반의 리랭킹
- 대규모 언어 모델(LLM)을 활용해 초기 검색 결과를 다시 평가하고, 사용자의 질문과 가장 관련성이 높은 순서로 결과를 정렬하는 방법
- 초기 검색 결과 준비(임베딩 모델 활용) > LLM을 통한 순위 조정 > 점수 기반 재정렬(기존 임베딩 모델의 검색 재정렬)
- 복잡한 쿼리나 미묘한 의미를 파악해야 하는 경우 효과적
- 장점 : 정확도 향상, 점수체계를 통해 객관적인 평가 기준을 적용할 수 있음
- 단점 : 비용/처리 시간 증가
7.3.1. 크로스인코더 기반 리랭킹
- LLM 기반보다는 리랭킹 선능이 다소 떨어질 수 있으나, 비용 효율성이 매우 우수
- BAAI/bge-reranker 리랭킹 특화 모델 사용
1) 이중 인코더
   - 텍스트를 벡터로 변환하여 의미적 유사도를 계산하는 방식
   - 질문 벡터와 문서 벡터 간의 코사인 유사도를 계산하여 가장 관련성이 높은 문서를 찾음
   - VectorStoreIndex, OpenAIEmbedding 
   - 장점 : 검색 속도가 매우 빠름
   - 단점 : 문맥을 충분히 반영하지 못함
2) 크로스 인코더
   - 질문과 문서를 따로 벡터화하지 않고 하나의 쌍으로 입력받아 직접적인 관련성을 판단하는 방식
   - 보다 정밀한 관련성 평가 가능
   - 문서 각각에 대해 연산을 수행해야 하므로 실시간 검색 시스템의 첫 단계 검색기로는 적합하지 않음
   - BAAI/bge-reranker-v2-m3
- 리랭크 : 이중 인코더의 속도 장점과 크로스 인코더의 정확성 장점 활용  
7.4 하이드(Hyde)
- 검색 시스템의 성능을 향상시키는 혁신적인 접근 방식(3단계)
- 가상 문서 생성 > 임베딩 및 검색 단계 > 최종 답변 생성 단계
- 장점 : 의미적 확장 능력, 검색 정확도 향상
- 단점 : 전통적인 검색 방식보다 많은 연산 비용 발생

리랭킹 : 초기 검색 결과를 더욱 정교하게 재정렬하여 사용자의 의도를 가장 부합하는 문서를 선별
하이드 : 가상의 이상적인 답변을 생성한 후, 이를 기반으로 검색의 정확도를 높임

In [ ]:
### 7.2 LLM기반의 리랭킹 ###
import requests
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryType
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine
from typing import List
from llama_index.core.schema import MetadataMode
import json

#분석할 PDF 파일을 웹에서 다운로드
url = "https://github.com/llama-index-tutorial/llama-index-tutorial/raw/main/ch07/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf"
filename = "2023_북한인권보고서.pdf"

response = requests.get(url)
with open(filename, "wb") as f:
    f.write(response.content)

print(f"{filename} 다운로드 완료")

2023_북한인권보고서.pdf 다운로드 완료


In [9]:
#라마인덱스의 핵심 설정: LLM, 임베딩 모델, 문서 분할 방식을 전역으로 설정
#GPT-4.1 언어 모델로 사용
Settings.llm = OpenAI(model="gpt-4-0125-preview", temperature=0.2)

#임베딩 모델 사용
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 300
Settings.chunk_overlap = 100

#PDF 문서를 읽고 벡터 인덱스 생성
reader = SimpleDirectoryReader(input_files=["2023_북한인권보고서.pdf"])

#문서에서 텍스트 추출
documents = reader.load_data()

#추출된 텍스트를 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

In [6]:
#리랭킹 구현
class DocumentScorer(BaseNodePostprocessor):
    #LLM을 사용해 문서의 관련성을 정밀하게 평가하고 점수를 매기는 클래스

    def evaluate_document(self, query: str, content: str) -> float:
        #LLM을 사용해 문서와 쿼리 간의 의미적 관련성을 1-10점으로 평가
        prompt = f"""
        아래 주어진 질문과 문서의 관련성을 평가해주세요.

        [평가 기준]
        - 문서가 질문에서 요구하는 정보를 직접적으로 포함하면 8-10점
        - 문서가 질문과 관련된 맥락을 포함하지만 직접적인 답이 아니면 4-7점
        - 문서가 질문과 거의 관련이 없으면 1-3점

        [주의사항]
        - 단순히 비슷한 단어가 등장하는 것은 높은 점수의 근거가 될 수 없습니다.
        - 질문의 의도와 문맥을 정확히 파악하여 평가해주세요.
        - 시간, 장소, 수치 등 구체적인 정보의 일치 여부를 중요하게 고려해주세요.

        질문 : {query}
        문서 : {content}

        응답은 반드시 다음 JSON 형식이어야 합니다:
        {{"relevance_score" : float}}
        """

        try:
            #LLM에 프롬프트를 전송하고 JSON 형식의 응답을 받음
            response = Settings.llm.complete(prompt)
            #응답에서 relevance_score 값 추출
            score = json.loads(response.text)["relevance_score"]
            #점수를 float로 변환하여 반환
            return float(score)
        except Exception as e:
            print(f"Error occurred:{str(e)}")
            return 5.0
        
    
    def _postprocess_nodes(self, nodes: List[NodeWithScore], query: QueryType) -> List[NodeWithScore]:
        #벡터 검색으로 찾은 4개 문서를 LLM으로 재평가하여 최적의 2개 선택
        print('\n===LLM이 4개의 검색 결과에 대해서 관련성을 평가합니다.===')
        score_docs = []
        for node in nodes:
            #현재 처리 중인 문서 노드에서 순수 텍스트 컨텐츠만 추출
            content = node.node.get_content(metadata_mode=MetadataMode.NONE)
            #LLM으로 문서 관련성 점수 계산(1-10점 사이)
            score = self.evaluate_document(str(query), content)
            #디버깅/모니터링을 위해 각 문서의 내용과 점수를 출력
            print(f"\nLLM 기반의 평가:\n{content}\n => 점수:{score}\n")
            #현재 노드와 계산된 점수를 튜플로 지정
            score_docs.append((node, score))

        #모든 문서를 점수 기준 내림차순으로 정렬하고 상위 2개만 선택하여 반환
        ranked_docs = sorted(score_docs, key=lambda x:x[1], reverse=True)
        return [node for nodes, _ in ranked_docs[:2]]

In [7]:
class SemanticRanker(BaseRetriever):
    #벡터 검색 결과에 LLM  기반 의미적 평가를 적용하여 최적의 문서를 선별하는 시스템

    def __init__(self, index, scorer):
        #생성자에서 벡터 검색용 인텍스와 LLM 기반 문서 평가기 인스턴스를 받아 저장
        self.index = index
        self.scorer = scorer

    def _retrieve(self, query: str) -> List[NodeWithScore]:
        #벡터 검색으로 유사도 기반 후보 문서 4개를 추출하고 LLM으로 재평가
        vector_results = self.index.as_retriever(similarity_top_k=4).retrieve(query)

        #초기 벡터 검색 결과를 디버깅/분석용으로 출력
        print("\n===실제 검색 결과(TOP-4)===")
        for i, node in enumerate(vector_results, 1):
            print(f"\n검색 문서{i}")
            print(node.node.get_content(metadata_mode=MetadataMode.NONE))

        #LLM으로 문서들을 재평가하고 재정렬하여 최적의 2개 선택
        reranked_results = self.scorer._postprocess_nodes(vector_results, query)

        #최종 선별된 문서를 디버깅/분석용으로 출력
        print("\n===LLM의 리랭킹 결과(TOP-2)===")
        for i, node in enumerate(reranked_results, 1):
            print(f"\n검색 문서{i}")
            print(node.node.get_content(metadata_mode=MetadataMode.NONE))
        
        return reranked_results

In [10]:
#문서 평가 및 검색 시스템 선언(초기화)
scorer = DocumentScorer() #LLM 기반 문서 평가기 생성
ranker = SemanticRanker(index, scorer) #벡터 검색과 LLM 평가를 결합한 시스템 생성
query_engine = RetrieverQueryEngine(retriever=ranker) #최종 질의응답 엔진 생성

#실제 쿼리 생성
query = "19년 말 평양시 소재 기업소에서 달마다 배급받은 음식"
print(f"\n질문:{query}")
response = query_engine.query(query) #쿼리 실행하여 응답 생성
print(f"\n최종 답:{response}")


질문:19년 말 평양시 소재 기업소에서 달마다 배급받은 음식

===실제 검색 결과(TOP-4)===

검색 문서1
대체로 합영·합작회사, 
외화벌이 기관 등 운영이 잘되는 경우였으며, 보수를 달러나 위안
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

검색 문서2
2023 북한인권보고서252며, 배급량의 80%는 강냉이로 쌀은 명절에만 배급되었다는 진술이 
있었다. 기업소에서 배급표는 매월 두 차례(7~8일경 및 21~22일경 
상·하순) 지급되었고, 거주지 배급소에서 식량으로 바꾸면 되었다고 
한다. 
식량배급이 되더라도 규정에 미치지 못하는 매우 적은 양을 받았
던 경우도 많았다.

검색 문서3
외화벌이 기관 등
에는 식량배급이 원활하게 이뤄지고 있었다는 증언이 수집되었다. 
2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀·
설탕·기름·야채·돼지고기 등을 배급받아 식량이 부족하지 않았다는 
증언과 2019년 중앙당 산하의 기업소에서 매월 쌀 6㎏ 정도, 기름 5
ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았
다는 증언이 있었다.

검색 문서4
2017년 
평양시 소재 기업소는 노동자에게 한번 1개월분의 식량을 배급하였
으며, 1인당 2㎏ 정도만 지급되었다는 진술이 있었다. 2018년 양강
도 철도노동자에게 직장노동자 700g으로 규정대로 배급표가 나왔
지만, 실제 식량배급은 1년 동안 감자 150㎏을 한번 받았다는 사례
와 2019년 시인민위원회 기관에서 일하던 가족이 1년에 한번 감자 
200kg을 배급받았지만, 운송비용 등으로 2만 원을 내야했다는 진
술도 있었다.

===LLM이 4개의 검색 결과에 대해서 관련성을 평가합니다.===

LLM 기반의 평가:
대체로 합영·합작회사, 
외화벌이 기관 등 운영이 잘되는 경우였으며, 보수를 달러나 위안
화 또는

In [ ]:
### 7.3 크로스 인코더 기반의 리랭킹 ###
import urllib.request
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.postprocessor import SentenceTransformerRerank

#기본 검색 엔진(리랭킹 없음)
basic_query_engine = index.as_query_engine(similarity_top_k=4)
#리랭킹 설정
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=2)
#리랭킹이 포함된 검색 엔진
rerank_query_engine = index.as_query_engine(similarity_top_k=4, node_postprocessors=[reranker])

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [14]:
#쿼리 실행
query = "19년 말 평양시 소재 기업소에서 달마다 배급받은 음식"

print("===기본 검색 엔진 검색 결과===")
basic_response = basic_query_engine.query(query)
print(f"\n질문:{query}")
print(f"\n답변:{basic_response.response}")
print("\n검색된 문서:")
for i, node in enumerate(basic_response.source_nodes):
    print(f"\n검색된 문서 {i+1}:")
    print(node.node.get_content())
    print("---")

===기본 검색 엔진 검색 결과===

질문:19년 말 평양시 소재 기업소에서 달마다 배급받은 음식

답변:2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀, 설탕, 기름, 야채, 돼지고기 등을 배급받아 식량이 부족하지 않았다고 증언했습니다. 또한, 2019년 중앙당 산하의 기업소에서는 매월 쌀 6㎏, 기름 5ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도를 받았다는 증언이 있었습니다.

검색된 문서:

검색된 문서 1:
대체로 합영·합작회사, 
외화벌이 기관 등 운영이 잘되는 경우였으며, 보수를 달러나 위안
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매
---

검색된 문서 2:
2023 북한인권보고서252며, 배급량의 80%는 강냉이로 쌀은 명절에만 배급되었다는 진술이 
있었다. 기업소에서 배급표는 매월 두 차례(7~8일경 및 21~22일경 
상·하순) 지급되었고, 거주지 배급소에서 식량으로 바꾸면 되었다고 
한다. 
식량배급이 되더라도 규정에 미치지 못하는 매우 적은 양을 받았
던 경우도 많았다.
---

검색된 문서 3:
외화벌이 기관 등
에는 식량배급이 원활하게 이뤄지고 있었다는 증언이 수집되었다. 
2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀·
설탕·기름·야채·돼지고기 등을 배급받아 식량이 부족하지 않았다는 
증언과 2019년 중앙당 산하의 기업소에서 매월 쌀 6㎏ 정도, 기름 5
ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았
다는 증언이 있었다.
---

검색된 문서 4:
2017년 
평양시 소재 기업소는 노동자에게 한번 1개월분의 식량을 배급하였
으며, 1인당 2㎏ 정도만 지급되었다는 진술이 있었다. 2018년 양강
도 철도노동자에게 직장노동자 700g으로 규정대로 배급표가 나왔
지만, 실제 식량배급은 1년 

In [16]:
print("\n\n===리랭킹 후 검색 결과===")
rerank_response = rerank_query_engine.query(query)
print(f"\n질문:{query}")
print(f"\n답변:{rerank_response.response}")
print("\n검색된 문서:")
for i, node in enumerate(rerank_response.source_nodes):
    print(f"\n검색된 문서 {i+1}:")
    print(node.node.get_content())
    print("---")



===리랭킹 후 검색 결과===

질문:19년 말 평양시 소재 기업소에서 달마다 배급받은 음식

답변:2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀, 설탕, 기름, 야채, 돼지고기 등을 배급받았으며, 중앙당 산하의 기업소에서는 매월 쌀 6㎏, 기름 5ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았다는 증언이 있었다.

검색된 문서:

검색된 문서 1:
외화벌이 기관 등
에는 식량배급이 원활하게 이뤄지고 있었다는 증언이 수집되었다. 
2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀·
설탕·기름·야채·돼지고기 등을 배급받아 식량이 부족하지 않았다는 
증언과 2019년 중앙당 산하의 기업소에서 매월 쌀 6㎏ 정도, 기름 5
ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았
다는 증언이 있었다.
---

검색된 문서 2:
2017년 
평양시 소재 기업소는 노동자에게 한번 1개월분의 식량을 배급하였
으며, 1인당 2㎏ 정도만 지급되었다는 진술이 있었다. 2018년 양강
도 철도노동자에게 직장노동자 700g으로 규정대로 배급표가 나왔
지만, 실제 식량배급은 1년 동안 감자 150㎏을 한번 받았다는 사례
와 2019년 시인민위원회 기관에서 일하던 가족이 1년에 한번 감자 
200kg을 배급받았지만, 운송비용 등으로 2만 원을 내야했다는 진
술도 있었다.
---


In [18]:
### 7.4 하이드 ###
import requests
import openai
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.llms.openai import OpenAI
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.embeddings.openai import OpenAIEmbedding

#기본 검색기 설정
retriever = VectorIndexRetriever(index=index, similarity_top_k=4)

#1단계. 가상 문서 생성
def generate_hypothetical_doc(question: str) -> str:
    """질문에 대한 가상의 이상적인 답변 문서 생성"""
    prompt = f"""
    주어진 질문에 대해, 마치 실제 문서에서 발췌한 것 같은 이상적인 답변을 작성해주세요.
    단, 구체적인 수치, 날짜, 트렌드와 같은 상세 정보를 포함해야 합니다.

    질문:{question}
    답변:"""

    response = Settings.llm.complete(prompt)
    return response.text

#2단계.임베딩 및 검색 단계
def search_with_hyde(hypothetical_doc: str):
    """가상 문서를 이용해 실제 문서 검색"""
    nodes = retriever.retrieve(hypothetical_doc)
    return [
        {
            'content' : node.node.get_content(),
            'score' : node.score
        } for node in nodes
    ]

#3단계. 최종 답변 생성 단계
def generate_final_answer(question:str, relevant_docs: list) -> str:
    """검색된 문서를 바탕으로 최종 답변 생성"""
    context = "\n\n".join([doc['content'] for doc in relevant_docs])

    prompt = f"""
    다음 검색 결과를 바탕으로 질문에 답변해주세요. 
    검색 결과의 정보를 최대한 사용하고, 없는 정보는 답변하지 마세요
    
    검색결과:{context}
    질문:{question}
    답변:"""

    response = Settings.llm.complete(prompt)
    return response.text

def process_query(question: str):
    """전체 HyDE 프로세스"""
    print("1.가상 문서 생성")
    hypothetical_doc = generate_hypothetical_doc(question)

    print("\n2.실제 문서 검색")
    relevant_docs = search_with_hyde(hypothetical_doc)

    print("\n3. 최종 답변 생성")
    final_answer = generate_final_answer(question, relevant_docs)
    
    return {
        "hypothetical_doc":hypothetical_doc,
        "relevant_docs":relevant_docs,
        "final_answer":final_answer   
    }

In [19]:
question = "북한에서 강제로 이루어지는 조직 생활은 무엇이 있나요?"
result = process_query(question)

print("\n===프로세스 결과===")
print("\n[가상 문서]:", result["hypothetical_doc"])
print("\n[실제 문서]")
for idx, doc in enumerate(result["relevant_docs"], 1):
    print(f"\문서 {idx} (유사도 점수: {doc['score']:4f}): {doc['content']}")
print("\n[최종 답변:]", result["final_answer"])

1.가상 문서 생성

2.실제 문서 검색

3. 최종 답변 생성

===프로세스 결과===

[가상 문서]: 북한에서 강제로 이루어지는 조직 생활은 국가의 체제 유지와 이념 전파를 목적으로 다양한 형태로 구성되어 있습니다. 이러한 조직 생활은 주민들의 일상에 깊숙이 침투하여, 개인의 사상과 행동을 통제하고 감시하는 역할을 합니다. 주요 조직 생활의 예는 다음과 같습니다:

1. **인민반 활동**: 북한의 기본적인 주거 단위인 인민반은 소규모의 지역 공동체로, 주민들은 정기적인 인민반 회의에 참여해야 합니다. 이 회의에서는 정치 교육, 주민 감시 및 서로에 대한 비판과 자기 비판이 이루어집니다. 인민반 활동은 주민들 사이의 상호 감시 체계를 강화하고, 정부에 대한 충성도를 높이는 데 목적이 있습니다.

2. **당 생활**: 조선로동당에 소속된 당원들은 정기적인 당 회의에 참석해야 하며, 이는 주로 정치 교육, 정책 전파, 당 지침에 대한 토론으로 구성됩니다. 당 생활은 당원들에게 높은 수준의 정치적 충성도와 활동 참여를 요구합니다.

3. **청년동맹 활동**: 조선사회주의청년동맹은 북한 청소년과 젊은이들을 대상으로 하는 조직으로, 정치 교육, 봉사 활동, 군사 훈련 등 다양한 활동을 통해 청년들을 사회주의 이념에 따라 교육하고 훈련합니다. 청년동맹 활동은 청소년기부터 국가 이념에 대한 충성심을 심어주는 역할을 합니다.

4. **사회적 대중운동**: 김일성, 김정일의 생일이나 국가의 중요한 기념일에 맞춰 전 국민이 참여하는 대규모 집회, 행진, 공연 등이 열립니다. 이러한 사회적 대중운동은 국가의 위업을 찬양하고, 주민들 사이에 단결과 애국심을 고취시키는 목적을 가집니다.

5. **노동단체 활동**: 노동자들은 노동당이 주관하는 노동단체 활동에 참여하게 됩니다. 이 활동은 노동자들의 정치 교육, 생산 목표 달성을 위한 동기 부여, 노동자들의 권익 보호를 목적으로 합니다. 하지만 실제로는 국가의 정책과 지침을 전파하고, 노동자들의 생각과 행동을 통제하는 수